# Data Visualisation: From Distribution to Business Story

**Type:** Original supplementary practice  
**Data:** Synthetic website and sales observations

The objective is to choose charts according to the analytical question, label them clearly, and avoid treating visual association as proof of causation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(8)
n = 800
df = pd.DataFrame({
    "channel": rng.choice(["Organic", "Paid Search", "Social", "Email"], n),
    "sessions": rng.integers(20, 900, n),
    "device": rng.choice(["Mobile", "Desktop", "Tablet"], n, p=[.62,.33,.05]),
    "campaign_spend": rng.gamma(3, 45, n)
})
channel_effect = df["channel"].map({"Organic":.035,"Paid Search":.052,"Social":.026,"Email":.061})
df["conversion_rate"] = np.clip(channel_effect + rng.normal(0,.012,n), .001, .15)
df["orders"] = rng.binomial(df["sessions"], df["conversion_rate"])
df["revenue"] = df["orders"] * rng.lognormal(4.1,.35,n)

df.head()

## 1. Distribution

A histogram is useful for understanding skew, spread and unusual values. The mean alone is not enough when a distribution is strongly right-skewed.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(df["revenue"], bins=30, edgecolor="white")
ax.axvline(df["revenue"].median(), linestyle="--", label="Median")
ax.axvline(df["revenue"].mean(), linestyle=":", label="Mean")
ax.set(title="Distribution of session-level revenue", xlabel="Revenue", ylabel="Frequency")
ax.legend()
plt.tight_layout()
plt.show()

print(df["revenue"].describe(percentiles=[.25,.5,.75,.9,.95]).round(2))

## 2. Compare categories

Use a box plot to compare spread and potential outliers. Use sample size and uncertainty alongside these summaries when communicating results.

In [ ]:
channel_order = ["Organic", "Paid Search", "Social", "Email"]
groups = [df.loc[df["channel"]==c, "conversion_rate"] for c in channel_order]

fig, ax = plt.subplots(figsize=(8,4))
ax.boxplot(groups, tick_labels=channel_order, showfliers=False)
ax.set(title="Conversion-rate distribution by channel", xlabel="Channel", ylabel="Conversion rate")
plt.tight_layout()
plt.show()

display(df.groupby("channel").agg(
    sessions=("sessions","sum"),
    orders=("orders","sum"),
    total_revenue=("revenue","sum"),
    mean_conversion_rate=("conversion_rate","mean")
).round(3))

## 3. Relationship between variables

A scatter plot can reveal patterns, clusters and heteroscedasticity. A relationship between spend and revenue does not, on its own, establish marketing incrementality.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
for channel, part in df.groupby("channel"):
    ax.scatter(part["campaign_spend"], part["revenue"], alpha=.45, s=18, label=channel)
ax.set(title="Campaign spend and session revenue", xlabel="Campaign spend (synthetic units)", ylabel="Revenue")
ax.legend()
plt.tight_layout()
plt.show()

print("Pearson correlation:", round(df["campaign_spend"].corr(df["revenue"]), 3))

## 4. Communicate one finding

A useful visualisation should make its scope and units explicit. Before sharing a chart, check the time period, denominator, missingness, scale, sample size and whether the audience may confuse association with causation.

**Practice task:** Replace the synthetic data with a permitted dataset, reproduce two charts, and write a three-sentence interpretation that includes one limitation.